# MiniGrid MPC comparison: H16/K8, H40/K20, and H100/K50

This notebook validates complete H16/K8 and H40/K20 experiments and combines H100/K50 base results with optional downloaded continuation results. A case is complete only with exactly ten unique episodes. The main table reports metrics only for baselines with all 25 cases complete; partial progress appears in the coverage table.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'wm').is_dir() and (candidate / 'trainer').is_dir():
            return candidate
    raise FileNotFoundError('Could not find repository root containing wm/ and trainer/')

REPO_ROOT = find_repo_root()
CONFIGS = {
    'H16 / K8': {
        'horizon': 16, 'execute_steps': 8,
        'csvs': [REPO_ROOT / 'wm/outputs/planning/mpc_baseline_target_eval/mpc_episode_results.csv'],
        'strict': True,
    },
    'H40 / K20': {
        'horizon': 40, 'execute_steps': 20,
        'csvs': [REPO_ROOT / 'outputs/planning/mpc_baseline_target_eval_h40_k20/mpc_episode_results.csv'],
        'strict': True,
    },
    'H100 / K50': {
        'horizon': 100, 'execute_steps': 50,
        'csvs': [REPO_ROOT / 'outputs/planning/mpc_baseline_target_eval_h100_k50/mpc_episode_results.csv'],
        'optional_globs': [
            'outputs/planning/mpc_baseline_target_eval_h100_k50_remaining/mpc_episode_results.csv',
            'outputs/planning/mpc_baseline_target_eval_h100_k50_server*/mpc_episode_results.csv',
        ],
        'strict': False,
    },
}
BASELINES = ['mac', 'target', 'dr', 'p2e']
DISPLAY_NAMES = {'mac': 'MAC', 'target': 'Target', 'dr': 'DR', 'p2e': 'P2E'}
TARGETS = [f'target_task{i}' for i in range(5)]
WM_SEEDS = list(range(5))
EPISODES_PER_CASE = 10
EXPECTED_CASES_PER_BASELINE = len(TARGETS) * len(WM_SEEDS)
EXPECTED_EPISODES_PER_BASELINE = EXPECTED_CASES_PER_BASELINE * EPISODES_PER_CASE
IDENTITY = ['baseline', 'wm_seed', 'target', 'episode']
REQUIRED = {
    'baseline', 'wm_seed', 'target', 'episode', 'success',
    'dense_return', 'environment_steps', 'imagined_transitions', 'execute_steps',
}

def source_paths(spec):
    paths = list(spec['csvs'])
    for pattern in spec.get('optional_globs', []):
        paths.extend(sorted(REPO_ROOT.glob(pattern)))
    return list(dict.fromkeys(path for path in paths if path.is_file()))

CONFIGS

{'H16 / K8': {'horizon': 16,
  'execute_steps': 8,
  'csvs': [PosixPath('/home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/wm/outputs/planning/mpc_baseline_target_eval/mpc_episode_results.csv')],
  'strict': True},
 'H40 / K20': {'horizon': 40,
  'execute_steps': 20,
  'csvs': [PosixPath('/home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/outputs/planning/mpc_baseline_target_eval_h40_k20/mpc_episode_results.csv')],
  'strict': True},
 'H100 / K50': {'horizon': 100,
  'execute_steps': 50,
  'csvs': [PosixPath('/home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/outputs/planning/mpc_baseline_target_eval_h100_k50/mpc_episode_results.csv')],
  'optional_globs': ['outputs/planning/mpc_baseline_target_eval_h100_k50_remaining/mpc_episode_results.csv',
   'outputs/planning/mpc_baseline_target_eval_h100_k50_server*/mpc_episode_results.csv'],
  'strict': False}}

In [2]:
def load_and_validate(label, spec):
    paths = source_paths(spec)
    if not paths:
        raise FileNotFoundError(f'{label}: no result CSV found')
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        missing = REQUIRED - set(frame.columns)
        if missing:
            raise ValueError(f'{label}: {path} missing columns {sorted(missing)}')
        frames.append(frame)
    frame = pd.concat(frames, ignore_index=True)
    frame = frame[
        frame['baseline'].isin(BASELINES)
        & frame['target'].isin(TARGETS)
        & frame['wm_seed'].isin(WM_SEEDS)
    ].copy()
    if frame.duplicated(IDENTITY).any():
        duplicate_rows = frame[frame.duplicated(IDENTITY, keep=False)]
        varying = duplicate_rows.groupby(IDENTITY, dropna=False).nunique(dropna=False).max(axis=1)
        if varying.gt(1).any():
            raise ValueError(f'{label}: conflicting duplicate episode identities found')
        frame = frame.drop_duplicates(IDENTITY, keep='last')
    if not frame['execute_steps'].eq(spec['execute_steps']).all():
        raise ValueError(f"{label}: execute_steps does not equal {spec['execute_steps']}")
    if 'horizon' in frame and not frame['horizon'].eq(spec['horizon']).all():
        raise ValueError(f"{label}: horizon does not equal {spec['horizon']}")
    case_counts = frame.groupby(['baseline', 'wm_seed', 'target']).size()
    incomplete = case_counts[~case_counts.eq(EPISODES_PER_CASE)]
    if spec['strict'] and not incomplete.empty:
        raise ValueError(f'{label}: cases must contain exactly 10 episodes: {incomplete.to_dict()}')
    if not spec['strict']:
        complete_cases = case_counts[case_counts.eq(EPISODES_PER_CASE)].index
        complete_index = pd.MultiIndex.from_frame(complete_cases.to_frame(index=False))
        frame_index = pd.MultiIndex.from_frame(frame[['baseline', 'wm_seed', 'target']])
        frame = frame.loc[frame_index.isin(complete_index)].copy()
    frame['success'] = frame['success'].astype(str).str.lower().eq('true')
    cases = (
        frame[['baseline', 'wm_seed', 'target']].drop_duplicates()
        .groupby('baseline').size()
        .reindex(BASELINES, fill_value=0)
    )
    if spec['strict']:
        expected_rows = len(BASELINES) * EXPECTED_EPISODES_PER_BASELINE
        if len(frame) != expected_rows or not cases.eq(EXPECTED_CASES_PER_BASELINE).all():
            raise ValueError(f'{label}: expected {expected_rows} rows and 25 cases per baseline; got {len(frame)} rows and {cases.to_dict()}')
    return frame, cases, paths

episode_data = {}
validation_rows = []
for label, spec in CONFIGS.items():
    frame, cases, paths = load_and_validate(label, spec)
    episode_data[label] = frame
    for baseline in BASELINES:
        count = int(cases.loc[baseline])
        validation_rows.append({
            'Configuration': label, 'Baseline': DISPLAY_NAMES[baseline],
            'Completed cases': count, 'Expected cases': EXPECTED_CASES_PER_BASELINE,
            'Episodes': count * EPISODES_PER_CASE,
            'Status': 'Complete' if count == EXPECTED_CASES_PER_BASELINE else 'Partial',
        })
validation = pd.DataFrame(validation_rows)
display(validation)
display(pd.DataFrame({
    label: {'rows': len(frame), 'cases': frame.groupby(['baseline', 'wm_seed', 'target']).ngroups,
            'source_files': len(source_paths(CONFIGS[label]))}
    for label, frame in episode_data.items()
}).T)

,Configuration,Baseline,Completed cases,Expected cases,Episodes,Status
0,H16 / K8,MAC,25,25,250,Complete
1,H16 / K8,Target,25,25,250,Complete
2,H16 / K8,DR,25,25,250,Complete
3,H16 / K8,P2E,25,25,250,Complete
4,H40 / K20,MAC,25,25,250,Complete
5,H40 / K20,Target,25,25,250,Complete
6,H40 / K20,DR,25,25,250,Complete
7,H40 / K20,P2E,25,25,250,Complete
8,H100 / K50,MAC,25,25,250,Complete
9,H100 / K50,Target,25,25,250,Complete


,rows,cases,source_files
H16 / K8,1000,100,1
H40 / K20,1000,100,1
H100 / K50,1000,100,3


In [3]:
METRICS = ['Success rate', 'Dense reward', 'Env steps', 'Imagined steps']
PLACEHOLDER = '—'
metric_columns = {
    'Success rate': 'success', 'Dense reward': 'dense_return',
    'Env steps': 'environment_steps', 'Imagined steps': 'imagined_transitions',
}
configuration_tables = {}
for label, frame in episode_data.items():
    case_counts = frame.groupby(['baseline', 'wm_seed', 'target']).size()
    completed = case_counts.groupby(level=0).size().reindex(BASELINES, fill_value=0).eq(EXPECTED_CASES_PER_BASELINE)
    aggregated = frame.groupby('baseline', sort=False).agg(
        **{metric: (column, 'mean') for metric, column in metric_columns.items()}
    ).reindex(BASELINES)[METRICS].astype(object)
    for baseline in BASELINES:
        if not completed.loc[baseline]:
            aggregated.loc[baseline, METRICS] = PLACEHOLDER
    configuration_tables[label] = aggregated

comparison = pd.concat(configuration_tables, axis=1)
comparison.index = [DISPLAY_NAMES[baseline] for baseline in BASELINES]
comparison.index.name = 'Baseline'

coverage = validation.pivot(index='Baseline', columns='Configuration', values='Completed cases')
coverage = coverage.reindex(index=[DISPLAY_NAMES[b] for b in BASELINES], columns=list(CONFIGS))
coverage = coverage.applymap(lambda count: f'{int(count)}/{EXPECTED_CASES_PER_BASELINE}')
coverage.index.name = 'Baseline'

formatters = {}
for config in CONFIGS:
    formatters[(config, 'Success rate')] = lambda value: value if isinstance(value, str) else f'{value:.1%}'
    formatters[(config, 'Dense reward')] = lambda value: value if isinstance(value, str) else f'{value:.3f}'
    formatters[(config, 'Env steps')] = lambda value: value if isinstance(value, str) else f'{value:.2f}'
    formatters[(config, 'Imagined steps')] = lambda value: value if isinstance(value, str) else f'{value:,.0f}'
styled_comparison = (
    comparison.style.format(formatters)
    .set_caption('Complete metrics only; H100/K50 partial baselines are shown as —')
    .set_properties(**{'text-align': 'right'})
)
display(styled_comparison)
display(coverage.style.set_caption('Completed cases / expected cases (25)'))

Configuration,H16 / K8,H40 / K20,H100 / K50
Baseline,,,
MAC,25/25,25/25,25/25
Target,25/25,25/25,25/25
DR,25/25,25/25,25/25
P2E,25/25,25/25,25/25


In [4]:
output_dir = REPO_ROOT / 'outputs/planning'
output_dir.mkdir(parents=True, exist_ok=True)
comparison_path = output_dir / 'mpc_h16k8_h40k20_h100k50_pivot.csv'
coverage_path = output_dir / 'mpc_h16k8_h40k20_h100k50_coverage.csv'
comparison.to_csv(comparison_path)
coverage.to_csv(coverage_path)
print(f'Saved: {comparison_path}')
print(f'Saved: {coverage_path}')

Saved: /home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/outputs/planning/mpc_h16k8_h40k20_h100k50_pivot.csv
Saved: /home/siyao/phd_file/Research/rlPractice/Curriculum_world_model_learning/outputs/planning/mpc_h16k8_h40k20_h100k50_coverage.csv
